# 🐄 Inteligencia de Mercado — Harina de Carne y Hueso (HCH)

Este notebook recopila, limpia y analiza datos de importación de insumos proteicos clave para la industria pecuaria colombiana, a partir de **dos fuentes**:

| Fuente | Descripción |
|--------|-------------|
| **Comtrade (ONU)** | API pública con datos anuales por país y partida arancelaria |
| **DIAN** | Archivos Excel con declaraciones de importación en Colombia |

### Partidas arancelarias monitoreadas

| Código | Producto |
|--------|----------|
| 230110 / 2301100000 | Harina de carne y hueso (HCH) |
| 230120 / 2301200000 | Harina de pescado |
| 120810 / 1208100000 | Harina de soya importada |
| 283526 / 2835260000 | Fosfato dicálcico |

---
> **Base de datos compartida:** `mercado_hch.db` (SQLite)  
> Todas las tablas se guardan en el mismo archivo para facilitar cruces posteriores.


In [1]:
## 0. Configuración global — EJECUTAR PRIMERO
import pandas as pd
import sqlite3
import glob
import os

DB_PATH          = "mercado_hch.db"
CARPETA = r"C:\Users\nvind\OneDrive\Documentos\PROYECTO\Portafolio\Proyecto hna_carna_hueso"
RUTA_ESAG = r"C:\Users\nvind\OneDrive\Documentos\PROYECTO\Portafolio\Proyecto hna_carna_hueso\series-hist-ESAG-Itrim2026.xls"

CARPETA_SCREENSHOTS = os.path.join(CARPETA, "screenshots")
csvs             = glob.glob(CARPETA + "\\*.csv")

os.makedirs(CARPETA_SCREENSHOTS, exist_ok=True)

print("✅ Configuración cargada")
print(f"   DB:          {DB_PATH}")
print(f"   Screenshots: {CARPETA_SCREENSHOTS}")
print(f"   CSVs:        {len(csvs)} archivos")

✅ Configuración cargada
   DB:          mercado_hch.db
   Screenshots: C:\Users\nvind\OneDrive\Documentos\PROYECTO\Portafolio\Proyecto hna_carna_hueso\screenshots
   CSVs:        2 archivos


---
## 1. Configuración general

Constantes y mapeo de partidas usados en ambas fuentes.


In [3]:
import requests
import pandas as pd
import sqlite3
import glob
from datetime import datetime

# ── Base de datos compartida ─────────────────
DB_PATH = "mercado_hch.db"

# ── Partidas arancelarias ────────────────────
# Formato corto (Comtrade)
PARTIDAS_COMTRADE = {
    "230110": "HCH",
    "230120": "harina_pescado",
    "120810": "harina_soya",
    "283526": "fosfato",
    '120190': "grano_soya",
    "023090": "harina_cordero"
    
}

# Formato largo (DIAN)
PARTIDAS_DIAN = [
    '2301100000',
    '2301200000',
    '1208100000',
    '2835260000',
    '1201900000',
    '0230901000'
    
]

NOMBRE_PARTIDA = {
    '2301100000': 'HCH',
    '2301200000': 'harina_pescado',
    '1208100000': 'harina_soya',
    '2835260000': 'fosfato',
    '1201900000':'grano_soya',
    '0230901000':'harina_cordero'
}

# ── Rango de precios válidos por producto (USD/ton) ─
RANGOS_PRECIO = {
    "HCH":            (100, 3000),
    "harina_pescado": (100, 5000),
    "harina_soya":  (100, 2000),
    "fosfato":        (100, 2000),
    'grano_soya':      (100, 1000),
    "harina_cordero":  (500, 3000)
}

print("✅ Configuración cargada correctamente.")

✅ Configuración cargada correctamente.


---
## 2. Fuente 1 — Comtrade (API de la ONU)

Consulta la API pública de UN Comtrade para obtener estadísticas anuales de importación hacia Colombia.

**Endpoint usado:** `https://comtradeapi.un.org/public/v1/preview/C/A/HS`  
**País reportante:** Colombia (código `170`)  
**Flujo:** M (importaciones)


### 2.1 Scraping — descarga de datos por partida


In [9]:
import time

PAISES_COMTRADE = {
    0: "Mundo (agregado)",
    32: "Argentina",
    68: "Bolivia",
    76: "Brasil",
    124: "Canadá",
    152: "Chile",
    156: "China",
    170: "Colombia",
    218: "Ecuador",
    246: "Finlandia",
    276: "Alemania",
    300: "Grecia",
    320: "Guatemala",
    380: "Italia",
    392: "Japón",
    440: "Lituania",
    484: "México",
    490: "Otros de Asia (no especificado)",
    504: "Marruecos",
    591: "Panamá",
    600: "Paraguay",
    604: "Perú",
    620: "Portugal",
    626: "Timor-Leste",
    643: "Federación Rusa",
    699: "India",
    724: "España",
    757: "Suiza",
    788: "Túnez",
    792: "Türkiye (Turquía)",
    838: "Zonas Francas",
    840: "Estados Unidos (código ISO estándar, no usado por Comtrade)",
    842: "Estados Unidos (código Comtrade — incluye Puerto Rico e Islas Vírgenes)",
    858: "Uruguay",
    862: "Venezuela",
    899: "Áreas no especificadas",
}
def scrape_comtrade_con_pais(cmd_code: str, nombre: str, años_lista: list) -> pd.DataFrame:
    url = "https://comtradeapi.un.org/public/v1/preview/C/A/HS"
    frames_año = []

    for año in años_lista:
        params = {
            "reporterCode": "170",
            "cmdCode":      cmd_code,
            "flowCode":     "M",
            "period":       año,
        }
        r = requests.get(url, params=params, timeout=30)
        if r.status_code == 429:
            time.sleep(10)
            r = requests.get(url, params=params, timeout=30)

        data = r.json().get("data", [])
        if data:
            frames_año.append(pd.DataFrame(data))
        else:
            print(f"    ⚠️  Sin datos para {nombre} en {año} (status {r.status_code})")
        time.sleep(2)

    if not frames_año:
        return pd.DataFrame()

    df = pd.concat(frames_año, ignore_index=True)
    cols = ['period', 'refYear', 'cmdCode', 'partnerCode',
            'cifvalue', 'netWgt', 'qty', 'qtyUnitAbbr']
    df = df[[c for c in cols if c in df.columns]].copy()

    df['pais_origen'] = df['partnerCode'].map(PAISES_ISO).fillna('Código ' + df['partnerCode'].astype(str))
    df['producto']      = nombre
    df['netWgt_ton']    = pd.to_numeric(df['netWgt'],    errors='coerce') / 1000
    df['cifvalue']      = pd.to_numeric(df['cifvalue'],  errors='coerce')
    df['precio_usd_ton'] = df['cifvalue'] / df['netWgt_ton']
    df['fecha_scraping'] = datetime.now().strftime('%Y-%m-%d')

    print(f"  ✅ {nombre}: {len(df)} registros | países distintos: {df['pais_origen'].nunique()}")
    return df

# ── Ejecutar para todas las partidas, 2020-2026 ──────────────────
print("🌐 Re-scrapeando Comtrade con país de origen...")
AÑOS_TODOS = ["2020","2021","2022","2023","2024","2025","2026"]

frames = []
for codigo, nombre in PARTIDAS_COMTRADE.items():
    df = scrape_comtrade_con_pais(codigo, nombre, AÑOS_TODOS)
    frames.append(df)
    time.sleep(2)

comtrade_con_pais = pd.concat(frames, ignore_index=True)
print(f"\nTotal registros: {len(comtrade_con_pais)}")

🌐 Re-scrapeando Comtrade con país de origen...
    ⚠️  Sin datos para HCH en 2026 (status 200)
  ✅ HCH: 659 registros | países distintos: 13
    ⚠️  Sin datos para harina_pescado en 2026 (status 200)
  ✅ harina_pescado: 291 registros | países distintos: 8
    ⚠️  Sin datos para harina_soya en 2020 (status 200)
    ⚠️  Sin datos para harina_soya en 2021 (status 200)
    ⚠️  Sin datos para harina_soya en 2026 (status 200)
  ✅ harina_soya: 76 registros | países distintos: 4
    ⚠️  Sin datos para fosfato en 2026 (status 200)
  ✅ fosfato: 924 registros | países distintos: 27
    ⚠️  Sin datos para grano_soya en 2026 (status 200)
  ✅ grano_soya: 255 registros | países distintos: 9
    ⚠️  Sin datos para harina_cordero en 2020 (status 200)
    ⚠️  Sin datos para harina_cordero en 2021 (status 200)
    ⚠️  Sin datos para harina_cordero en 2022 (status 200)
    ⚠️  Sin datos para harina_cordero en 2023 (status 200)
    ⚠️  Sin datos para harina_cordero en 2024 (status 200)
    ⚠️  Sin datos pa

In [15]:
PAISES_COMTRADE = {
    0: "Mundo (agregado)",
    32: "Argentina",
    68: "Bolivia",
    76: "Brasil",
    124: "Canadá",
    152: "Chile",
    156: "China",
    170: "Colombia",
    218: "Ecuador",
    246: "Finlandia",
    276: "Alemania",
    300: "Grecia",
    320: "Guatemala",
    380: "Italia",
    392: "Japón",
    440: "Lituania",
    484: "México",
    490: "Otros de Asia (no especificado)",
    504: "Marruecos",
    591: "Panamá",
    600: "Paraguay",
    604: "Perú",
    620: "Portugal",
    626: "Timor-Leste",
    643: "Federación Rusa",
    699: "India",
    724: "España",
    757: "Suiza",
    788: "Túnez",
    792: "Türkiye (Turquía)",
    838: "Zonas Francas",
    840: "Estados Unidos (código ISO estándar, no usado por Comtrade)",
    842: "Estados Unidos (código Comtrade — incluye Puerto Rico e Islas Vírgenes)",
    858: "Uruguay",
    862: "Venezuela",
    899: "Áreas no especificadas",
}

In [17]:
comtrade_full['pais_origen'] = comtrade_full['partnerCode'].map(PAISES_COMTRADE).fillna('Código ' + comtrade_full['partnerCode'].astype(str))

print("Distribución final de países por producto:")
print(comtrade_full.groupby(['producto', 'pais_origen']).size().reset_index(name='n').sort_values(['producto','n'], ascending=[True, False]).to_string(index=False))

Distribución final de países por producto:
      producto                                                             pais_origen   n
           HCH                                                        Mundo (agregado) 218
           HCH                                                                  Brasil  80
           HCH                                                                  España  69
           HCH                                                                  Italia  63
           HCH                                                               Argentina  60
           HCH                                                                Paraguay  55
           HCH Estados Unidos (código Comtrade — incluye Puerto Rico e Islas Vírgenes)  50
           HCH                                                                 Uruguay  16
           HCH                                                  Áreas no especificadas  16
           HCH                                 

In [19]:
PAISES_COMTRADE.update({251: "Francia", 458: "Malasia", 703: "Eslovaquia", 752: "Suecia"})
comtrade_full['pais_origen'] = comtrade_full['partnerCode'].map(PAISES_COMTRADE).fillna('Código ' + comtrade_full['partnerCode'].astype(str))

con = sqlite3.connect(DB_PATH)
comtrade_full.to_sql("importaciones", con, if_exists="replace", index=False)
con.close()
print(f"✅ 'importaciones' actualizada con país de origen — {len(comtrade_full)} registros")

✅ 'importaciones' actualizada con país de origen — 2205 registros


In [21]:
# ── Solo filtro de integridad — sin nulos, sin peso/valor en cero o negativo ──
comtrade_sin_nulos = comtrade_full[
    (comtrade_full['netWgt_ton'] > 0) &
    (comtrade_full['cifvalue']   > 0) &
    (comtrade_full['precio_usd_ton'].notna())
].copy()

print(f"Registros originales : {len(comtrade_full)}")
print(f"Sin nulos/inválidos  : {len(comtrade_sin_nulos)}")

con = sqlite3.connect(DB_PATH)
comtrade_sin_nulos.to_sql("importaciones", con, if_exists="replace", index=False)
con.close()

print(f"✅ Tabla 'importaciones' guardada — {len(comtrade_sin_nulos)} registros")
print(f"   Columnas: {list(comtrade_sin_nulos.columns)}")

Registros originales : 2205
Sin nulos/inválidos  : 2181
✅ Tabla 'importaciones' guardada — 2181 registros
   Columnas: ['period', 'refYear', 'cmdCode', 'partnerCode', 'cifvalue', 'netWgt', 'qty', 'qtyUnitAbbr', 'pais_origen', 'producto', 'netWgt_ton', 'precio_usd_ton', 'fecha_scraping']


---
## 3. Fuente 2 — DIAN (archivos locales)

Procesa archivos Excel descargados del sistema de la DIAN con declaraciones de importación.

**Flujo de trabajo:**
1. Detectar archivos `.xlsx` en la carpeta del proyecto
2. Convertir cada Excel a CSV (operación pesada, solo se hace una vez)
3. Leer en chunks y filtrar por partida arancelaria
4. Calcular precio USD/ton y COP/ton
5. Guardar en SQLite


### 3.1 Detectar archivos Excel


In [16]:
# Ajusta esta ruta a la carpeta donde tienes los archivos DIAN
archivos = glob.glob(CARPETA + "\\*.xlsx") + glob.glob(CARPETA + "\\*.xls")
print(f"📁 Archivos encontrados: {len(archivos)}")
for a in archivos:
    print(f"  • {a}")

📁 Archivos encontrados: 0


### 3.2 Convertir Excel a CSV

> ⏳ Esta operación puede tardar varios minutos por archivo.  
> Solo es necesario hacerla **una vez** — los CSV quedan guardados para reusos futuros.


In [18]:
def excel_a_csv(ruta_excel: str) -> str:
    """
    Convierte un archivo Excel a CSV (UTF-8 con BOM para Excel en Windows).
    Salta archivos ESAG — ya están cargados en SQLite.
    Retorna la ruta del CSV generado.
    """
    # Saltar ESAG — ya está en la base de datos como sacrificio_bovino
    if 'ESAG' in ruta_excel:
        print(f"  ⏭️  Saltando {ruta_excel.split(chr(92))[-1]} — ya en SQLite")
        return None

    # Convertir según formato
    ruta_csv = ruta_excel.replace(".xlsx", ".csv").replace(".xls", ".csv")
    print(f"Convirtiendo: {ruta_excel.split(chr(92))[-1]} ...")

    try:
        if ruta_excel.endswith('.xls'):
            df = pd.read_excel(ruta_excel, dtype=str, engine='xlrd')
        else:
            df = pd.read_excel(ruta_excel, dtype=str, engine='openpyxl')

        df.to_csv(ruta_csv, index=False, encoding='utf-8-sig')
        print(f"  ✅ Listo. Filas: {len(df):,} | Guardado en: {ruta_csv}")
        return ruta_csv

    except Exception as e:
        print(f"  ⚠️ Error en {ruta_excel.split(chr(92))[-1]}: {e}")
        return None

### 3.3 Lectura en chunks — inspección inicial

Antes de procesar en firme, se hace una lectura rápida del primer archivo para validar que las columnas de partida arancelaria estén presentes.


In [37]:
def leer_dian_chunked(ruta_csv: str, chunksize: int = 50_000) -> pd.DataFrame:
    """
    Lee un CSV de DIAN por chunks y retiene solo filas de las partidas de interés.
    Útil para inspección inicial o archivos muy grandes.
    """
    print(f"Leyendo: {ruta_csv.split(chr(92))[-1]}")
    chunks_filtrados = []

    for chunk in pd.read_csv(
        ruta_csv, dtype=str, chunksize=chunksize, encoding='utf-8-sig'
    ):
        chunk.columns = chunk.columns.str.strip().str.upper()

        # Detectar columna de partida arancelaria
        col_partida = [c for c in chunk.columns if 'SUBPARTIDA' in c or 'PARTIDA' in c]
        if not col_partida:
            print("  ❌ No se encontró columna de partida. Columnas disponibles:")
            print(chunk.columns.tolist())
            break

        col_partida = col_partida[0]
        filtrado = chunk[chunk[col_partida].str.strip().isin(PARTIDAS_DIAN)]
        if len(filtrado) > 0:
            chunks_filtrados.append(filtrado)

    if not chunks_filtrados:
        print("  ⚠️  Sin registros para las partidas buscadas.")
        return pd.DataFrame()

    resultado = pd.concat(chunks_filtrados, ignore_index=True)
    print(f"  ✅ Filas encontradas: {len(resultado):,}")
    return resultado


# Inspeccionar el primer CSV
df_prueba = leer_dian_chunked(csvs[0])

if not df_prueba.empty:
    print("Columnas disponibles:")
    print(df_prueba.columns.tolist())
    print("Primeras 3 filas:")
    print(df_prueba.head(3))

Leyendo: 01_Importaciones_2026_Enero.csv
  ✅ Filas encontradas: 75
Columnas disponibles:
['BANC_CODIGO_BANCO', 'COD_ADUANA_PRESENTADA', 'ADUANA_PRESENTADA', 'CODIGO_SUCURSAL', 'CODIGO_CAJERO', 'CONSECUTIVO_CAJERO', 'DIGITO_VERIFICACION_DOC', 'CODIGO_MODALIDAD_DECLARAC', 'COD_TIPO_DECLARACION', 'TIPO_DECLARACION', 'NUM_DECLARACION_ANT', 'FECHA_DECLARACION_ANT', 'MANIFIESTO_DE_CARGA', 'FECHA_MANIFIESTO', 'DOCUMENTO_TRANSPORTE', 'MODA_CODIGO_MODALIDAD', 'CODIGO_DEPOSITO', 'RAZON_SOCIAL_DECLARANTE', 'TIPO_IDENTIFICAC_DECLARAN', 'NIT_DECLARANTE', 'DIGITO_VERIFI_NIT_DECLARA', 'NOMBRE_IMPORTADOR', 'TIPO_IDENTIFICAC_IMPORTAD', 'NIT_IMPORTADOR', 'DIGITO_VERIFI_NIT_IMPORTA', 'CIUDAD_PAIS_IMPORTADOR', 'DIRECCION_IMPORTADOR', 'NOMBRE_EXPORTADOR', 'TIPO_IDENTIFICAC_EXPORTAD', 'NUMERO_IDENTIFICAC_EXPORT', 'DIGITO_VERIFI_NIT_EXPORTA', 'DIRECCION_EXPORTADOR', 'CIUDAD_PAIS_EXPORTADOR', 'ACTIVIDAD_ECONOMICA_SEC', 'CODIGO_PAIS_PROCEDENCIA', 'PAIS_PROCEDENCIA', 'COD_MODO_TRANSPORTE', 'MODO_TRANSPORTE', 'C

### 3.4 Procesamiento completo — cálculo de precios


In [39]:
# Columnas de interés del archivo DIAN
COLS_UTILES = [
    'SUBPARTIDA_ARANCELARIA',
    'FECHA_ACEPTACION_DECLARACION',
    'PAIS_ORIGEN',
    'PAIS_PROCEDENCIA',
    'PESO_NETO',
    'VALOR_CIF_USD',
    'VALOR_FOB_USD',
    'VALOR_FLETES_USD',
    'DEPARTAMENTO_DESTINO',
    'MODO_TRANSPORTE',
    'NOMBRE_IMPORTADOR',
    'DESCRIPCION_MERCANCIA',
    'TASA_CAMBIO',
]


def procesar_dian(ruta_csv: str, chunksize: int = 50_000) -> pd.DataFrame:
    """
    Lee un CSV de DIAN por chunks, filtra partidas de interés,
    calcula precio USD/ton y COP/ton, y retorna el DataFrame limpio.
    """
    print(f"Procesando: {ruta_csv.split(chr(92))[-1]}")
    chunks = []

    for chunk in pd.read_csv(
        ruta_csv, dtype=str, chunksize=chunksize, encoding='utf-8-sig'
    ):
        chunk.columns = chunk.columns.str.strip().str.upper()
        col_p = [c for c in chunk.columns if 'SUBPARTIDA' in c][0]
        filtrado = chunk[chunk[col_p].str.strip().isin(PARTIDAS_DIAN)].copy()

        if not filtrado.empty:
            cols = [c for c in COLS_UTILES if c in filtrado.columns]
            chunks.append(filtrado[cols])

    if not chunks:
        print("  ⚠️  Sin registros.")
        return pd.DataFrame()

    df = pd.concat(chunks, ignore_index=True)

    # Convertir tipos numéricos
    df['PESO_NETO']     = pd.to_numeric(df['PESO_NETO'],     errors='coerce')
    df['VALOR_CIF_USD'] = pd.to_numeric(df['VALOR_CIF_USD'], errors='coerce')
    df['TASA_CAMBIO']   = pd.to_numeric(df['TASA_CAMBIO'],   errors='coerce')

    # Calcular precio por tonelada (USD y COP)
    df['PESO_TON']       = df['PESO_NETO'] / 1000
    df['PRECIO_USD_TON'] = df['VALOR_CIF_USD'] / df['PESO_TON']
    df['PRECIO_COP_TON'] = df['PRECIO_USD_TON'] * df['TASA_CAMBIO']

    # Fecha y etiqueta de producto
    df['FECHA']   = pd.to_datetime(
        df['FECHA_ACEPTACION_DECLARACION'], errors='coerce', dayfirst=True)
    df['PRODUCTO'] = df['SUBPARTIDA_ARANCELARIA'].str.strip().map(NOMBRE_PARTIDA)
    df['FUENTE']   = 'DIAN'

    print(f"  ✅ Filas procesadas: {len(df):,}")
    return df


# Procesar todos los CSVs disponibles
frames_dian = []
for csv in csvs:
    df = procesar_dian(csv)
    if not df.empty:
        frames_dian.append(df)

dian_final = pd.concat(frames_dian, ignore_index=True)

print("📊 Resumen por producto (DIAN):")
print(
    dian_final.groupby('PRODUCTO')[['PRECIO_USD_TON', 'PRECIO_COP_TON', 'PESO_TON']]
    .agg({'PRECIO_USD_TON': 'mean', 'PRECIO_COP_TON': 'mean', 'PESO_TON': 'sum'})
    .round(1)
)

Procesando: 01_Importaciones_2026_Enero.csv


C:\Users\nvind\AppData\Local\Temp\ipykernel_8460\622304819.py:55: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['FECHA']   = pd.to_datetime(


  ✅ Filas procesadas: 75
Procesando: 02_Importaciones_2026_Febrero.csv
  ✅ Filas procesadas: 62
📊 Resumen por producto (DIAN):
            PRECIO_USD_TON  PRECIO_COP_TON  PESO_TON
PRODUCTO                                            
fosfato             9397.7      34565135.5    3888.9
grano_soya           534.1       1965566.6  102399.1


C:\Users\nvind\AppData\Local\Temp\ipykernel_8460\622304819.py:55: UserWarning: Parsing dates in %Y%m%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['FECHA']   = pd.to_datetime(


### 3.5 Guardar en SQLite


In [41]:
con = sqlite3.connect(DB_PATH)
dian_final.to_sql("dian_2026", con, if_exists="replace", index=False)
con.close()

print(f"✅ Tabla 'dian_2026' guardada en {DB_PATH}")

✅ Tabla 'dian_2026' guardada en mercado_hch.db


### 3.6 Diagnóstico — verificar lo que entró a la base

Útil para detectar partidas que no coinciden, valores en cero o registros mal filtrados.


In [43]:
con = sqlite3.connect(DB_PATH)
df_diag = pd.read_sql("SELECT * FROM dian_2026", con)
con.close()

print("📋 Subpartidas encontradas y su frecuencia:")
print(df_diag['SUBPARTIDA_ARANCELARIA'].value_counts())

print("📏 Detalle de peso, valor y precio por registro:")
print(
    df_diag[['SUBPARTIDA_ARANCELARIA', 'PESO_TON', 'VALOR_CIF_USD', 'PRECIO_USD_TON']]
    .to_string()
)

📋 Subpartidas encontradas y su frecuencia:
SUBPARTIDA_ARANCELARIA
2835260000    69
1201900000    68
Name: count, dtype: int64
📏 Detalle de peso, valor y precio por registro:
    SUBPARTIDA_ARANCELARIA    PESO_TON  VALOR_CIF_USD  PRECIO_USD_TON
0               2835260000   108.00000       84672.00      784.000000
1               2835260000     4.80000        8675.73     1807.443750
2               2835260000    27.00000       20588.40      762.533333
3               2835260000   108.00000       89054.29      824.576759
4               2835260000    54.00000       41176.80      762.533333
5               2835260000    54.00000       41176.80      762.533333
6               2835260000    27.00000       21798.64      807.357037
7               2835260000    54.00000       44038.60      815.529630
8               2835260000    81.00000       62451.00      771.000000
9               2835260000    54.00000       44848.76      830.532593
10              2835260000    26.00000       25367.75   

---
## 4. Resumen de tablas en la base de datos

Consulta rápida para confirmar qué tablas existen y cuántos registros tiene cada una.


In [45]:
con = sqlite3.connect(DB_PATH)

tablas = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'", con
)['name'].tolist()

print(f"📦 Tablas en {DB_PATH}:")
for tabla in tablas:
    n = pd.read_sql(f"SELECT COUNT(*) AS n FROM {tabla}", con).iloc[0, 0]
    print(f"  • {tabla}: {n:,} registros")

con.close()

📦 Tablas en mercado_hch.db:
  • importaciones: 1,316 registros
  • sacrificio_bovino: 210 registros
  • ingredientes_comparacion: 8 registros
  • importaciones_limpias: 1,119 registros
  • dian_2026: 137 registros


# 5. DANE ESAG — Series históricas sacrificio bovino y bufalino

In [7]:
!pip install xlrd


In [9]:
import pandas as pd
import sqlite3

DB_PATH = "mercado_hch.db"

def parsear_fecha_esag(valor):
    """
    Maneja dos formatos de fecha del DANE:
    - Formato Excel: 2008-10-01 (antes de 2020)
    - Formato texto: ene-2020pr (desde 2020)
    """
    meses_es = {
        'ene':1,'feb':2,'mar':3,'abr':4,
        'may':5,'jun':6,'jul':7,'ago':8,
        'sep':9,'oct':10,'nov':11,'dic':12
    }
    try:
        return pd.to_datetime(valor)
    except:
        try:
            texto = str(valor).strip().lower().replace('pr','').strip()
            partes = texto.split('-')
            if len(partes) == 2:
                mes     = meses_es.get(partes[0].strip())
                año_str = partes[1].strip()
                if mes and año_str.isdigit() and len(año_str) == 4:
                    return pd.Timestamp(year=int(año_str), month=mes, day=1)
            return pd.NaT
        except:
            return pd.NaT

def parsear_numero_esag(valor):
    """
    Maneja números con punto como separador de miles.
    290.251 → 290251
    """
    """
    Maneja tres formatos numéricos del DANE:
    - Entero normal:        369016       → 369016
    - Float imputado:       366419.1352  → 366419  (redondear)
    - Miles español:        290.251      → 290251  (quitar punto)
    """
    try:
        if pd.isna(valor):
            return None
        texto = str(valor).strip()

        if any(c.isalpha() for c in texto):
            return None

        if texto in ('', 'nan'):
            return None

        if '.' in texto:
            decimales = len(texto.split('.')[-1])

            if decimales == 3:
                # Separador de miles español: 290.251 → 290251
                return int(texto.replace('.', ''))
            else:
                # Float real con muchos decimales: 366419.135 → 366419
                return int(round(float(texto)))
        else:
            return int(float(texto))

    except (ValueError, TypeError):
        return None

def cargar_esag_robusto(ruta, sheet, col_nombre):
    df_raw = pd.read_excel(
        ruta, sheet_name=sheet,
        header=None, engine='xlrd'
    )
    datos = df_raw.iloc[10:, [0, 1]].copy()
    datos.columns = ['fecha_raw', col_nombre]
    datos = datos[datos['fecha_raw'].notna()].copy()
    datos['fecha'] = datos['fecha_raw'].apply(parsear_fecha_esag)
    datos[col_nombre] = datos[col_nombre].apply(parsear_numero_esag)
    datos = datos[
        datos['fecha'].notna() &
        datos[col_nombre].notna()
    ][['fecha', col_nombre]].reset_index(drop=True)
    return datos

# ── Cargar vacunos y bufalinos ────────────────────────────────
vacunos   = cargar_esag_robusto(RUTA_ESAG, 'Cuadro 1', 'vacunos_cabezas')
bufalinos = cargar_esag_robusto(RUTA_ESAG, 'Cuadro 2', 'bufalinos_cabezas')

# ── Merge y calcular potencial HCH ───────────────────────────
df_esag = pd.merge(vacunos, bufalinos, on='fecha', how='outer')
df_esag = df_esag.sort_values('fecha').reset_index(drop=True)
df_esag['potencial_hch'] = (
    df_esag['vacunos_cabezas'].fillna(0) +
    df_esag['bufalinos_cabezas'].fillna(0)
)
df_esag['fuente'] = 'DANE_ESAG'

print(f"✅ Datos cargados: {len(df_esag)} meses")
print(f"   Rango: {df_esag['fecha'].min().date()} → {df_esag['fecha'].max().date()}")
print(f"\n📊 Últimos 6 meses:")
print(df_esag.tail(6).to_string(index=False))
print(f"\n📈 Estadísticas potencial HCH (cabezas/mes):")
print(df_esag['potencial_hch'].describe().round(0))

# ── Guardar en SQLite ─────────────────────────────────────────
con = sqlite3.connect(DB_PATH)
df_esag.to_sql("sacrificio_bovino", con, if_exists="replace", index=False)
con.close()
print(f"\n✅ Tabla 'sacrificio_bovino' guardada en {DB_PATH}")

✅ Datos cargados: 210 meses
   Rango: 2008-10-01 → 2026-03-01

📊 Últimos 6 meses:
     fecha  vacunos_cabezas  bufalinos_cabezas  potencial_hch    fuente
2025-10-01         300068.0             4785.0       304853.0 DANE_ESAG
2025-11-01         278812.0             4105.0       282917.0 DANE_ESAG
2025-12-01         308978.0             3769.0       312747.0 DANE_ESAG
2026-01-01         283171.0             5372.0       288543.0 DANE_ESAG
2026-02-01         270673.0             5031.0       275704.0 DANE_ESAG
2026-03-01         289616.0             5314.0       294930.0 DANE_ESAG

📈 Estadísticas potencial HCH (cabezas/mes):
count       210.0
mean     301240.0
std       32470.0
min      211727.0
25%      279429.0
50%      296150.0
75%      326824.0
max      369022.0
Name: potencial_hch, dtype: float64

✅ Tabla 'sacrificio_bovino' guardada en mercado_hch.db


# 5.2 DANE ESAG — Series históricas sacrificio Cerdos

In [13]:
import pandas as pd
hojas = pd.ExcelFile(RUTA_ESAG, engine='xlrd').sheet_names
print(hojas)

['Contenido', 'Cuadro 1', 'Cuadro 2', 'Cuadro 3', 'Cuadro 4', 'Cuadro 5']


In [15]:
# ── Cargar porcinos (misma función robusta ya definida arriba) ──
porcinos = cargar_esag_robusto(RUTA_ESAG, 'Cuadro 3', 'porcinos_cabezas')

# ── Unir a la serie ya existente ─────────────────────────────
df_esag = pd.merge(df_esag, porcinos, on='fecha', how='outer')
df_esag = df_esag.sort_values('fecha').reset_index(drop=True)

print(f"✅ Porcinos cargados: {len(porcinos)} meses")
print(f"   Rango: {porcinos['fecha'].min().date()} → {porcinos['fecha'].max().date()}")
print(df_esag.tail(6).to_string(index=False))

# ── Volver a guardar sacrificio_bovino con la columna nueva ───
# (el nombre de tabla se queda igual para no romper el ETL existente;
# 'potencial_hch' no cambia, sigue siendo solo vacunos+bufalinos)
con = sqlite3.connect(DB_PATH)
df_esag.to_sql("sacrificio_bovino", con, if_exists="replace", index=False)
con.close()
print(f"\n✅ Tabla 'sacrificio_bovino' actualizada con porcinos_cabezas")

✅ Porcinos cargados: 210 meses
   Rango: 2008-10-01 → 2026-03-01
     fecha  vacunos_cabezas  bufalinos_cabezas  potencial_hch    fuente  porcinos_cabezas
2025-10-01         300068.0             4785.0       304853.0 DANE_ESAG          598513.0
2025-11-01         278812.0             4105.0       282917.0 DANE_ESAG          562236.0
2025-12-01         308978.0             3769.0       312747.0 DANE_ESAG          702414.0
2026-01-01         283171.0             5372.0       288543.0 DANE_ESAG          566027.0
2026-02-01         270673.0             5031.0       275704.0 DANE_ESAG          538695.0
2026-03-01         289616.0             5314.0       294930.0 DANE_ESAG          593273.0

✅ Tabla 'sacrificio_bovino' actualizada con porcinos_cabezas


# 6. TRM

In [36]:
import requests
import pandas as pd
import sqlite3

DB_PATH = "mercado_hch.db"

# ── Descargar TRM desde datos.gov.co ─────────────────────────────
url = "https://www.datos.gov.co/api/views/mcec-87by/rows.csv?accessType=DOWNLOAD"

print("Descargando TRM desde datos.gov.co...")
df_trm = pd.read_csv(url)

# Ver columnas disponibles
print("Columnas:", df_trm.columns.tolist())
print(df_trm.head(3))

Descargando TRM desde datos.gov.co...
Columnas: ['VALOR', 'UNIDAD', 'VIGENCIADESDE', 'VIGENCIAHASTA']
     VALOR UNIDAD VIGENCIADESDE VIGENCIAHASTA
0  3205.80    COP    28/07/2026    28/07/2026
1  3210.56    COP    25/07/2026    27/07/2026
2  3219.31    COP    24/07/2026    24/07/2026


In [38]:
# ── Limpiar y guardar TRM en DB ───────────────────────────────────
df_trm.columns = ['trm_cop_usd', 'unidad', 'fecha_desde', 'fecha_hasta']

# Usar fecha_desde como fecha de referencia
df_trm['fecha'] = pd.to_datetime(df_trm['fecha_desde'], format='%d/%m/%Y')
df_trm = df_trm[['fecha', 'trm_cop_usd']].copy()
df_trm = df_trm.sort_values('fecha').reset_index(drop=True)

# Filtrar rango relevante para el proyecto
df_trm = df_trm[df_trm['fecha'] >= '2022-01-01']

# Rellenar fines de semana y festivos (forward fill)
fecha_completa = pd.date_range(
    start=df_trm['fecha'].min(),
    end=df_trm['fecha'].max(),
    freq='D'
)
df_trm = df_trm.set_index('fecha').reindex(fecha_completa).ffill().reset_index()
df_trm.columns = ['fecha', 'trm_cop_usd']
df_trm['fecha'] = df_trm['fecha'].dt.strftime('%Y-%m-%d')

# Guardar en DB
with sqlite3.connect(DB_PATH) as con:
    df_trm.to_sql('trm_diaria', con, if_exists='replace', index=False)
    print(f"✅ Tabla 'trm_diaria' guardada: {len(df_trm)} días")
    print(f"   Rango: {df_trm['fecha'].min()} → {df_trm['fecha'].max()}")
    print(f"   TRM promedio: {df_trm['trm_cop_usd'].mean():.0f} COP/USD")
    print(df_trm.tail(5).to_string(index=False))

✅ Tabla 'trm_diaria' guardada: 1667 días
   Rango: 2022-01-04 → 2026-07-28
   TRM promedio: 4105 COP/USD
     fecha  trm_cop_usd
2026-07-24      3219.31
2026-07-25      3210.56
2026-07-26      3210.56
2026-07-27      3210.56
2026-07-28      3205.80


# 7. CME (CHICAGO MERCANTILE EXCHANGE)

In [7]:
!pip install yfinance

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------------------- -------------- 1.0/1.7 MB 6.3 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 5.3 MB/s eta 0:00:00
  Attempting uninstall: cffi
    Found existing installation: cffi 1.17.1
    Uninstalling cffi-1.17.1:
      Successfully uninstalled cffi-1.17.1


In [2]:
import yfinance as yf
import sqlite3
import pandas as pd
from datetime import date

DB_PATH = "mercado_hch.db"
print("Descargando CME Harina de Soya y Grano de Maíz...")
soya = yf.download("ZM=F", start="2022-01-01", end=date.today().isoformat(),
                   auto_adjust=True, progress=False)
maiz = yf.download("ZC=F", start="2022-01-01", end=date.today().isoformat(),
                   auto_adjust=True, progress=False)

soya_close = soya[('Close', 'ZM=F')]
maiz_close = maiz[('Close', 'ZC=F')]

df_cme = pd.DataFrame({
    'fecha':                        soya_close.index.strftime('%Y-%m-%d'),
    'harina_soya_cme_usd_ton':      soya_close.values,
    'grano_maiz_cme_usd_bushel':    maiz_close.values
})

# ── Conversiones correctas ────────────────────────────────────────
df_cme['harina_soya_cme_usd_ton']  = (df_cme['harina_soya_cme_usd_ton'] / 0.907185).round(2)
df_cme['grano_maiz_cme_usd_ton']   = (df_cme['grano_maiz_cme_usd_bushel'] / 100 / 25.4 * 1000).round(2)

# ── Guardar solo el diario (fuente única) ──────────────────────────
with sqlite3.connect(DB_PATH) as con:
    df_cme[['fecha', 'harina_soya_cme_usd_ton', 'grano_maiz_cme_usd_ton']].to_sql(
        'cme_diario', con, if_exists='replace', index=False
    )
    print(f"✅ Tabla 'cme_diario' guardada: {len(df_cme)} días")
    print(f"   Rango: {df_cme['fecha'].min()} → {df_cme['fecha'].max()}")

Descargando CME Harina de Soya y Grano de Maíz...
✅ Tabla 'cme_diario' guardada: 1146 días
   Rango: 2022-01-03 → 2026-07-28


# 8. NOA FENÓMENO DEL NIÑO

In [43]:
# 8. 
import pandas as pd
import requests
import io
import sqlite3

DB_PATH = "mercado_hch.db"

# ── Descargar RONI directo de NOAA ──────────────────────────────────
URL_RONI = "https://www.cpc.ncep.noaa.gov/data/indices/RONI.ascii.txt"

SEASON_TO_MONTH = {
    'DJF': 1, 'JFM': 2, 'FMA': 3, 'MAM': 4,
    'AMJ': 5, 'MJJ': 6, 'JJA': 7, 'JAS': 8,
    'ASO': 9, 'SON': 10, 'OND': 11, 'NDJ': 12,
}

try:
    resp = requests.get(URL_RONI, timeout=15)
    resp.raise_for_status()
    df_roni_raw = pd.read_csv(io.StringIO(resp.text), sep=r"\s+")
    print(f"✅ RONI descargado en vivo de NOAA — {len(df_roni_raw)} trimestres "
          f"({df_roni_raw['YR'].min()}-{df_roni_raw['YR'].max()})")
except Exception as e:
    print(f"⚠️  No se pudo descargar ({e}). Revisa tu conexión a internet.")
    raise

# ── Cada trimestre se asigna a su mes central (ver explicación anterior) ──
df_roni_raw['mes']      = df_roni_raw['SEAS'].map(SEASON_TO_MONTH)
df_roni_raw['anio_mes'] = df_roni_raw['YR'].astype(str) + '-' + df_roni_raw['mes'].astype(str).str.zfill(2)
df_roni = df_roni_raw[['anio_mes', 'ANOM']].rename(columns={'ANOM': 'roni'})

con = sqlite3.connect(DB_PATH)
df_roni.to_sql("roni_mensual", con, if_exists="replace", index=False)
con.close()

print(f"✅ Tabla 'roni_mensual' guardada — última fecha disponible: {df_roni['anio_mes'].max()}")
print("ℹ️  Esta celda se puede volver a correr cada mes y siempre traerá el dato más reciente.")


✅ RONI descargado en vivo de NOAA — 917 trimestres (1950-2026)
✅ Tabla 'roni_mensual' guardada — última fecha disponible: 2026-05
ℹ️  Esta celda se puede volver a correr cada mes y siempre traerá el dato más reciente.


# 9. INGREDIENTES

In [35]:
import pandas as pd
import sqlite3

RUTA_FEDNA = r"C:\Users\nvind\OneDrive\Documentos\PROYECTO\Portafolio\Proyecto hna_carna_hueso\composicion_nutricional_FEDNA.xlsx"

df_fedna = pd.read_excel(RUTA_FEDNA, sheet_name="Composicion FEDNA")
print(f"✅ {len(df_fedna)} insumos cargados desde FEDNA")
print(df_fedna[['Insumo (BD proyecto)', 'Proteina bruta (%)', 'P total (%)', 
                 'P disponible (%)', 'Pdig Porcino (%)', 'Pdig Aves (%)']].to_string(index=False))

✅ 10 insumos cargados desde FEDNA
 Insumo (BD proyecto)  Proteina bruta (%)  P total (%)  P disponible (%)  Pdig Porcino (%)  Pdig Aves (%)
                  HCH                49.3         3.85              3.46              2.73           2.39
        harina_sangre                87.0         0.21              0.21              0.17           0.17
  harina_pluma_sangre                83.9         0.60              0.60              0.48           0.48
      harina_visceras                61.8         0.72              0.65              0.53           0.48
         harina_hueso                 8.0        15.00               NaN               NaN            NaN
subproductos_carnicos                49.3         3.85              3.46              2.73           2.39
 torta_soya_importada                44.0         0.61              0.19              0.24           0.26
           grano_soya                36.8         0.56              0.18              0.22           0.24
       harin

In [37]:
cDB_PATH = "mercado_hch.db"
con = sqlite3.connect(DB_PATH)

precio_bmc = pd.read_sql("""
    SELECT b.producto AS insumo, b.precio_usd_ton AS precio_usd_ton_bmc, b.fecha AS fecha_precio_bmc
    FROM bmc_precios b
    INNER JOIN (
        SELECT producto, MAX(fecha) AS max_fecha
        FROM bmc_precios
        WHERE producto IN ('HCH','harina_sangre','harina_pluma_sangre','harina_visceras',
                            'harina_hueso','subproductos_carnicos','torta_soya_importada',
                            'fosfato_monodicalcico')
        GROUP BY producto
    ) ult ON b.producto = ult.producto AND b.fecha = ult.max_fecha
""", con)

precio_comtrade_raw = pd.read_sql("""
    SELECT producto AS insumo, refYear, precio_usd_ton
    FROM importaciones
    WHERE pais_origen = 'Mundo (agregado)'
      AND producto IN ('grano_soya','harina_pescado','HCH')
""", con)
ultimo_anio_por_insumo = precio_comtrade_raw.groupby('insumo')['refYear'].max().reset_index()
precio_comtrade_raw = precio_comtrade_raw.merge(ultimo_anio_por_insumo, on=['insumo','refYear'])
precio_comtrade = precio_comtrade_raw.groupby(['insumo','refYear'])['precio_usd_ton'].median().round(1).reset_index()
precio_comtrade.columns = ['insumo', 'anio_precio_comtrade', 'precio_usd_ton_comtrade']

NO_PAISES = ['Mundo (agregado)', 'Áreas no especificadas', 'Zonas Francas', 'Otros de Asia (no especificado)']
df_ct = pd.read_sql("SELECT * FROM importaciones", con)
df_ct_paises = df_ct[~df_ct['pais_origen'].isin(NO_PAISES) & (df_ct['refYear'] == 2025)]

disponibilidad = df_ct_paises.groupby('producto')['pais_origen'].nunique().reset_index()
disponibilidad.columns = ['insumo', 'n_paises_proveedores']

def concentracion(grupo):
    total = grupo['netWgt_ton'].sum()
    return round(grupo.groupby('pais_origen')['netWgt_ton'].sum().max() / total * 100, 1) if total > 0 else None

riesgo = df_ct_paises.groupby('producto').apply(concentracion, include_groups=False).reset_index()
riesgo.columns = ['insumo', 'pct_concentracion_top_pais']

con.close()

df_comparacion = df_fedna.rename(columns={'Insumo (BD proyecto)': 'insumo'})
df_comparacion = df_comparacion.merge(precio_bmc, on='insumo', how='left')
df_comparacion = df_comparacion.merge(precio_comtrade, on='insumo', how='left')
df_comparacion = df_comparacion.merge(disponibilidad, on='insumo', how='left')
df_comparacion = df_comparacion.merge(riesgo, on='insumo', how='left')

# ── Guardar en la BD ──
con = sqlite3.connect(DB_PATH)
df_comparacion.to_sql("ingredientes_comparacion", con, if_exists="replace", index=False)
con.close()
print(f"✅ ingredientes_comparacion actualizada en la BD — {len(df_comparacion)} insumos, {len(df_comparacion.columns)} columnas")
print(df_comparacion.columns.tolist())

✅ ingredientes_comparacion actualizada en la BD — 10 insumos, 17 columnas
['insumo', 'Categoria FEDNA', 'Proteina bruta (%)', 'P total (%)', 'P disponible (%)', 'Digestibilidad Porcino (%)', 'Digestibilidad Aves (%)', 'Pdig Porcino (%)', 'Pdig Aves (%)', 'Notas', 'Fuente (link)', 'precio_usd_ton_bmc', 'fecha_precio_bmc', 'anio_precio_comtrade', 'precio_usd_ton_comtrade', 'n_paises_proveedores', 'pct_concentracion_top_pais']


In [39]:
con = sqlite3.connect(DB_PATH)
df_comparacion.to_sql("ingredientes_comparacion", con, if_exists="replace", index=False)
con.close()
print(f"✅ ingredientes_comparacion guardada — {len(df_comparacion)} insumos")

✅ ingredientes_comparacion guardada — 10 insumos


# 10. DEMANDA (CONCENTRADOS)

In [14]:
import sqlite3
import pandas as pd

DB_PATH = "mercado_hch.db"

with sqlite3.connect(DB_PATH) as con:
    resultado = pd.read_sql("""
        SELECT producto, COUNT(*) AS filas, MIN(fecha) AS desde, MAX(fecha) AS hasta,
               ROUND(SUM(cantidad_kg)/1000, 1) AS ton_total
        FROM bmc_precios_limpio
        WHERE producto LIKE '%perro%' OR producto LIKE '%gato%' OR producto LIKE '%mascota%' OR producto LIKE '%concentrado%'
        GROUP BY producto
    """, con)

print(resultado)

             producto  filas       desde       hasta  ton_total
0    concentrado_aves    648  2022-09-12  2026-05-26  4648204.6
1  concentrado_cerdos    666  2022-09-12  2026-05-27  3361183.5
2   concentrado_gatos    780  2022-09-12  2026-05-29   157827.6
3  concentrado_perros    717  2022-09-12  2026-05-29   492306.4
